In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
!pip install openpyxl
import openpyxl

import warnings
warnings.filterwarnings("ignore")

In [ ]:
tickers = pd.read_excel('nifty500_universe_classified.xlsx', index_col=0)

In [ ]:
# input
start = '2012-01-01' 
end = '2026-01-01' 

In [ ]:
print("Parsing Started....")
count=0
for i in tickers.Symbol:
  symbol = i+ '.NS'
  df = yf.download(symbol,start,end)
  df.to_csv("NIFTY500/"+str(count)+ "_" + i +'.csv')
  print("Done ", count, " ", symbol)
  count+=1

  # if count==3:
  #   break
  print("PARSED!")

In [ ]:
import yfinance as yf
import pandas as pd

rows = []

for t in tickers:
    try:
        info = yf.Ticker(t).info

        rows.append({
            "ticker": t,
            "company_name": info.get("longName", "Unknown"),
            "sector": info.get("sector", "Unknown"),
            "market_cap": info.get("marketCap", None)
        })

        print("Done:", t)

    except Exception as e:
        print("Failed:", t, e)

In [ ]:

import os
import glob

# Define your input and output directories
input_folder = "NIFTY500"
output_folder = "dataset500"

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Find all CSV files in the input folder
csv_files = glob.glob(f"{input_folder}/*.csv")

print(f"Found {len(csv_files)} files. Starting cleanup...\n")

for file in csv_files:
    filename = os.path.basename(file)
    output_path = os.path.join(output_folder, filename)
    
    try:
        # yfinance saves CSVs with a 2-level header (Price, Ticker)
        # header=[0, 1] reads the first two rows as column names
        # index_col=0 sets the first column ('Date') as the index
        df = pd.read_csv(file, header=[0, 1], index_col=0)
        
        # Flatten the MultiIndex columns to just the first level (Open, High, etc.)
        # This removes the 'AARTIIND.NS' ticker level completely
        df.columns = df.columns.get_level_values(0)
        
        # Ensure the index is named 'Date'
        df.index.name = 'date'
        
        # Define the desired OHLCV column order
        # (We check against df.columns to avoid errors if a column is missing)
        desired_order = ['Open', 'High', 'Low', 'Close', 'Volume']
        final_columns = [col for col in desired_order if col in df.columns]
        
        # Reorder the dataframe
        df = df[final_columns]
        
        # Drop rows where all OHLCV values are NaN (cleans up any empty artifact rows)
        df = df.dropna(how='all')
        
        # Save to the clean folder
        df.to_csv(output_path)
        print(f"Cleaned and saved: {filename}")
        
    except Exception as e:
        print(f"ERROR cleaning {filename}: {e}")

print("\nCleanup Completed.")

In [ ]:
import os
import shutil


BASE_DATASET = "dataset500"
OUT_DIR = "classified_dataset"

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Size folders
for size in ["large", "mid", "small"]:
    os.makedirs(os.path.join(OUT_DIR, "size", size), exist_ok=True)

# Sector folders
for sector in df["sector"].dropna().unique():
    os.makedirs(os.path.join(OUT_DIR, "sector", sector), exist_ok=True)

def clean_sector(sector):
    return sector.replace("/", "_").strip()

for _, row in df.iterrows():
    ticker = row["ticker"]
    size = row["size"]
    sector = row["sector"]
    clean_sector(sector)
    src = os.path.join(BASE_DATASET, f"{ticker}.csv")

    if not os.path.exists(src):
        continue

    # Copy to size folder
    dst_size = os.path.join(OUT_DIR, "size", size, f"{ticker}.csv")
    shutil.copy2(src, dst_size)

    # Copy to sector folder
    if pd.notna(sector):
        dst_sector = os.path.join(OUT_DIR, "sector", sector, f"{ticker}.csv")
        shutil.copy2(src, dst_sector)